# Finish E4.4: XLM-R large emotion, seeds 21 and 42

Seed 13 is already in the repository. This notebook trains the two missing seeds with the same code (commit `fe90b1d`, the commit seed 13 used) and downloads `e4_4_seeds_21_42.zip` at the end.

**How to run**
1. Runtime > Change runtime type > **T4 GPU** > Save.
2. Runtime > **Run all**. Allow Google Drive access when asked.
3. Keep this tab open for about 40 minutes (about 16 minutes per seed plus setup).
4. The last cell downloads the zip. Send that zip back.

If Colab disconnects, reconnect and **Run all** again: a seed that already finished is copied to Drive and skipped.

If `freeze_splits.py --check` fails, stop: the data split differs from the one every other model used.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'
assert torch.cuda.get_device_properties(0).total_memory > 14e9, 'Need a 15 GB or larger GPU (T4 or better)'

In [ ]:
# Google Drive keeps finished seeds safe if the session drops
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE = '/content/drive/MyDrive/mhrisk_e44'
os.makedirs(SAVE, exist_ok=True)
# train on the fast local disk; only finished seeds go to Drive
os.environ['MHRISK_RUNS_DIR'] = '/content/runs'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
EXP = 'E4_4_xlmr_large_emotion'

In [ ]:
COMMIT = 'fe90b1d1c07795f57fb32d6b0bc9102dadcd7d81'  # same code as seed 13
if not os.path.exists('/content/repo'):
    !git clone -q https://github.com/faizdevx/Multilingual-Mental-Health-Risk-.git /content/repo
%cd /content/repo
!git checkout -q {COMMIT} && git log --oneline -1
!pip install -q -r requirements-colab.txt 2>&1 | tail -n 2

In [ ]:
!python scripts/download_data.py
!python scripts/prepare_data.py > /dev/null
!python scripts/freeze_splits.py --check

In [ ]:
import shutil, subprocess, pathlib
for seed in (21, 42):
    saved = pathlib.Path(SAVE) / EXP / f'seed_{seed}'
    if (saved / 'metrics.json').exists():
        print(f'seed {seed}: already finished, skipping')
        continue
    print(f'seed {seed}: training (about 16 min on a T4)...', flush=True)
    r = subprocess.run(['python', 'scripts/run_experiment.py', f'configs/{EXP}.json', '--seeds', str(seed), '--device', 'cuda'],
                       capture_output=True, text=True)
    print('\n'.join(r.stdout.splitlines()[-6:]))
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise SystemExit(f'seed {seed} failed (exit {r.returncode})')
    local = pathlib.Path('/content/runs') / EXP / f'seed_{seed}'
    assert (local / 'metrics.json').exists(), f'seed {seed}: no metrics.json written'
    if saved.exists():
        shutil.rmtree(saved)
    shutil.copytree(local, saved, ignore=shutil.ignore_patterns('resume*', 'checkpoint'))
    print(f'seed {seed}: saved to Drive')

In [ ]:
# check both seeds, show their test scores
import json
for seed in (21, 42):
    m = json.load(open(f'{SAVE}/{EXP}/seed_{seed}/metrics.json'))
    t = m['test']['emotion']
    print(seed, 'GPU', m.get('device'), '| test macro-F1', round(t['macro_f1'], 4), '| by language', {k: round(v['macro_f1'], 4) for k, v in t['by_language'].items()})

## Download the results
Metrics and per-example probabilities only; no text, no model weights.

In [ ]:
!cd {SAVE} && rm -f e4_4_seeds_21_42.zip && zip -qr e4_4_seeds_21_42.zip {EXP}/seed_21 {EXP}/seed_42 -x '*/resume*' && unzip -l e4_4_seeds_21_42.zip | tail -n 3
from google.colab import files
files.download(f'{SAVE}/e4_4_seeds_21_42.zip')